# conv-windowing-1d — faded example 2: Output width for unfold-based 1-D conv

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-1d`. Running the beacon reports progress on the `CNN: 1-D conv windowing` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: 1-D conv windowing` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-windowing-1d`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-windowing-1d"
DD_SUBTOPIC = "CNN: 1-D conv windowing"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`Tensor.unfold(dimension, size, step)` builds a sliding-window view. For a stride-`step` 1-D conv over width `W` with kernel width `KW`, the number of windows (the output width) is `OW = (W - KW) // step + 1`. The test reshapes/contracts the unfolded view, so it needs this count to be correct.

## Faded exercise 2

Complete `conv1d_unfold_and_count(x, KW, step)`. It must unfold `x` along its last dimension and also return the expected output width `OW`. The unfold call is given; you must compute `OW` from `W`, `KW`, and `step`. The test checks both the view shape and that contracting against a kernel matches `F.conv1d(x, weight, stride=step)`.

**Fill in:** the output width OW = (W - KW) // step + 1.

In [ ]:
import torch as t
import torch.nn.functional as F
from einops import einsum

def conv1d_unfold_and_count(x: t.Tensor, KW: int, step: int):
    B, IC, W = x.shape
    OW = (W - KW) // step + 1
    windows = x.unfold(dimension=-1, size=KW, step=step)
    return windows, OW


def _test():
    t.manual_seed(0)
    x = t.randn(2, 4, 15)
    weight = t.randn(6, 4, 3)
    KW, step = 3, 2
    windows, OW = conv1d_unfold_and_count(x, KW, step)
    assert OW == (15 - KW) // step + 1, OW
    assert tuple(windows.shape) == (2, 4, OW, KW), windows.shape
    out = einsum(windows, weight, 'b i o k, c i k -> b c o')
    ref = F.conv1d(x, weight, stride=step)
    assert t.allclose(out, ref, atol=1e-4), (out - ref).abs().max().item()


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn.functional as F
from einops import einsum

def conv1d_unfold_and_count(x: t.Tensor, KW: int, step: int):
    B, IC, W = x.shape
    OW = (W - KW) // step + 1
    windows = x.unfold(dimension=-1, size=KW, step=step)
    return windows, OW
```
</details>